# Banco de pruebas Codewars: Who is the killer? (`7 kyu`)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maximofernandezriera/katas-colab/blob/main/notebooks/kata-who-is-the-killer-1.ipynb)

Notebook generado con `kata2notebook.py` a partir de
[https://www.codewars.com/kata/5f709c8fb0d88300292a7a9d](https://www.codewars.com/kata/5f709c8fb0d88300292a7a9d).

> Trabaja el kata aquí antes de enviarlo a Codewars. La API pública no
> incluye el código inicial ni los tests oficiales: en el Paso 2 defines
> la firma de la función (mírala en la web del kata) y en el Paso 3 tus
> propios tests.


In [ ]:
# PASO 0 — ejecuta esta celda para cargar las utilidades
# ============================================================
#  BANCO DE PRUEBAS — framework de tests estilo Codewars
#  (no hace falta modificar esta celda, solo ejecutarla)
# ============================================================
import time
import difflib
import urllib.request
import json
import traceback

def cargar_kata(slug_o_id):
    """Descarga los datos del kata desde la API pública de Codewars."""
    url = f"https://www.codewars.com/api/v1/code-challenges/{slug_o_id}"
    with urllib.request.urlopen(url) as r:
        return json.load(r)

def mostrar_kata(kata):
    from IPython.display import Markdown, display
    rank = kata.get("rank") or {}
    display(Markdown(
        f"## {kata['name']}  ·  `{rank.get('name', '?')}`\n"
        f"**Enlace:** {kata['url']}  \n"
        f"**Tags:** {', '.join(kata.get('tags', []))}\n\n---\n\n"
        + kata.get("description", "")
    ))

# ---- mini framework estilo Codewars ----
_fallos = []
_aciertos = []

def _reset():
    _fallos.clear()
    _aciertos.clear()

def describe(nombre):
    print(f"\n== {nombre} ==")

def it(nombre):
    print(f"  · {nombre}")

def assert_equals(real, esperado, msg=""):
    if real == esperado:
        _aciertos.append(1)
        print(f"    ✔ OK  →  {real!r}")
    else:
        _fallos.append((real, esperado, msg))
        print(f"    ✘ FALLO  →  obtenido {real!r}  esperado {esperado!r}  {msg}")

def resumen():
    total = len(_aciertos) + len(_fallos)
    print(f"\n{'='*40}\nResultado: {len(_aciertos)}/{total} tests superados")
    if total == 0:
        print("⚠ Faltan pruebas: escribe tus tests en el Paso 3 con assert_equals(...).")
    elif _fallos:
        print("¡Sigue intentándolo! Revisa los fallos de arriba.")
    else:
        print("¡Todos los tests superados! 🎉 Ahora compara tu solución abajo.")

def comparar_con_referencia(mi_funcion, referencias, casos):
    """Ejecuta mi_funcion y cada función de referencia sobre 'casos'
    (lista de tuplas de argumentos) y compara resultados y tiempos."""
    for nombre, src in referencias.items():
        ns = {}
        exec(src, ns)
        ref = ns.get("solucion") or next(iter(ns.values()))
        print(f"\n### {nombre}")
        for caso in casos:
            args = caso if isinstance(caso, tuple) else (caso,)
            try:
                t0 = time.perf_counter()
                esperado = ref(*args)
                t_ref = (time.perf_counter() - t0) * 1e6
            except Exception as e:
                print(f"  referencia falla en {args}: {e}")
                continue
            try:
                t0 = time.perf_counter()
                obtenido = mi_funcion(*args)
                t_mio = (time.perf_counter() - t0) * 1e6
            except Exception:
                print(f"  tu solución lanza excepción en {args}:")
                traceback.print_exc(limit=1)
                continue
            marca = "✔" if obtenido == esperado else "✘"
            print(f"  {marca} {args} → tuyo={obtenido!r} ref={esperado!r}  "
                  f"(tú: {t_mio:.0f}µs / ref: {t_ref:.0f}µs)")

def diff(mi_src, ref_src, nombre="referencia"):
    """Muestra un diff línea a línea entre tu código y otra solución."""
    d = difflib.unified_diff(
        ref_src.splitlines(), mi_src.splitlines(),
        fromfile=nombre, tofile="tu solución", lineterm="")
    print("\n".join(d))


In [ ]:
# PASO 1 — carga el enunciado del kata
kata = cargar_kata("who-is-the-killer-1")
mostrar_kata(kata)

## Enunciado en castellano (traducción)

> Traducción al castellano del enunciado; el original en inglés se muestra en el Paso 1.

### ¿Quién es el asesino?

¡Han matado a varias personas! Has conseguido reducir los sospechosos a unos pocos. Por suerte, conoces a todas las personas que cada sospechoso vio el día de los asesinatos.

**Tarea.** Recibes un diccionario con los nombres de los sospechosos y las personas que cada uno vio ese día, que puede tener esta pinta:

```python
{'James': ['Jacob', 'Bill', 'Lucas'],
 'Johnny': ['David', 'Kyle', 'Lucas'],
 'Peter': ['Lucy', 'Kyle']}
```

y también una lista con los nombres de las víctimas:

```python
['Lucas', 'Bill']
```

Devuelve el nombre del único asesino: en este caso `'James'`, porque es la única persona que vio tanto a `'Lucas'` como a `'Bill'`.

## Hoja de ruta — lee el kata antes de programar

Rellena esto **antes** de escribir código (hay una versión imprimible en `guia-lectura-kata.html`):

1. **¿Qué recibo?** Nº de parámetros y tipos (número, texto, booleano…):
2. **¿Qué devuelvo?** ¿Número, texto, booleano? ¿El texto debe ser exacto?
3. **Ejemplo, con mis palabras:** coge un ejemplo del enunciado y explica por qué da ese resultado.
4. **Casos límite:** ¿qué pasa con 0, negativos, cadena vacía, valores iguales…?
5. **El diccionario:** ¿qué guardo como clave y qué como valor? ¿tengo que crearlo, recorrerlo o consultarlo? ¿qué pasa si una clave todavía no existe?
6. **Plan:** escribe los pasos en castellano, uno por línea, sin código.
7. **Después:** ¿mis tests cubren esos casos límite? ¿Qué aprendí?


## Paso 1.5 · Mi plan (modo pensar)

**Piensa antes de programar.** Rellena esta plantilla en lenguaje natural o
pseudocódigo — es parte de la entrega y tu profesor la revisará.

**ENTRADA:** ¿qué recibo? (parámetros y tipos, en mis palabras)
> …

**PLAN:** pasos numerados, cada uno empezando por verbo (*leer*, *comprobar*,
*devolver*, *calcular*…). Mínimo 3 pasos.
> 1. …
> 2. …
> 3. …

**SALIDA:** ¿qué devuelvo exactamente?
> …

**CASOS QUE VIGILO:** los casos límite que detecté en la hoja de ruta
> …


In [ ]:
# PASO 1.5 (continuación) — tu plan en formato revisable
# Escribe cada paso de tu plan como una cadena. La celda comprueba
# que el plan existe y tiene al menos 3 pasos (no corrige el contenido).

mi_plan = [
    # "1. leer ...",
    # "2. comprobar ...",
    # "3. devolver ...",
]

if not mi_plan:
    print("⚠ Tu plan está vacío: rellena la lista mi_plan con tus pasos.")
elif len(mi_plan) < 3:
    print(f"⚠ Solo {len(mi_plan)} paso(s): detalla el plan un poco más.")
else:
    print(f"✔ Plan registrado: {len(mi_plan)} pasos.")
    for paso in mi_plan:
        print("   ", paso)


In [ ]:
# PASO 2 — TU SOLUCIÓN
# Copia la firma de la función desde codewars.com

def solucion():
    raise NotImplementedError('escribe aquí tu solución')


## Escalera de ayudas (úsala solo si te atascas 15 min)

Regla: baja un peldaño por vez — Nivel 1 → Nivel 2 → Nivel 3. Nunca saltes al último sin leer el primero.

<details>
<summary><b>Nivel 1 — Pista de clase</b></summary>

Recorre los pares clave-valor con `suspects.items()`: la clave es el sospechoso y el valor, la lista de personas que vio. El asesino es quien vio a **todas** las víctimas: comprueba que cada nombre de `dead` está en su lista. `all(...)` con un generador encaja muy bien.
</details>

<details>
<summary><b>Nivel 2 — Pseudocódigo</b></summary>

```
para cada (nombre, vistos) en suspects.items():
    si TODOS los muertos están en vistos:
        devolver nombre
```
</details>

<details>
<summary><b>Nivel 3 — Esqueleto con huecos</b> (rellena los `___`)</summary>

```python
def killer(suspects, dead):
    for name, seen in suspects.___():
        if all(d in ___ for d in dead):
            return ___
```
</details>


In [ ]:
# PASO 3 — TUS TESTS
_reset()

describe("Ejemplos del enunciado")
it("caso 1")
# assert_equals(solucion(...), valor_esperado)

describe("Casos límite")
it("...")
# assert_equals(solucion(...), ...)

resumen()


In [ ]:
# ============================================================
#  PASO 4 — Compara tu solución con otras
#  Tras resolver el kata en codewars.com se desbloquean las
#  soluciones de la comunidad: pégalas aquí para analizarlas.
#  Tu profesor también puede darte soluciones de referencia.
# ============================================================

# Tu código (pégalo tal cual para el diff):
mi_codigo = """
def solucion(...):
    pass  # pega aquí tu solución
"""

# Soluciones con las que comparar (nombre: código).
# Deben definir una función llamada 'solucion' (o ser la única función).
soluciones_referencia = {
    "solución del profesor": """
def solucion(...):
    pass  # solución de referencia
""",
}

# Los mismos casos de prueba de la celda anterior:
casos = [
    # (argumentos...) — rellénalos igual que en tus tests
]

# Descomenta cuando tengas algo que comparar:
# comparar_con_referencia(solucion, soluciones_referencia, casos)
# diff(mi_codigo, list(soluciones_referencia.values())[0])


---
### Cómo usar este banco de pruebas
1. **Lee el enunciado** y abre el kata en codewars.com.
2. **Escribe tu solución** en la celda correspondiente.
3. **Completa los tests**: añade casos de ejemplo del enunciado y casos límite que se te ocurran. Escribir buenos tests es parte del ejercicio.
4. **Resuelve el kata en la web** con tu solución cuando pase tus tests.
5. **Compara**: al resolverlo, Codewars desbloquea las soluciones de la comunidad. Pega las que te interesen en la última celda y analiza qué hacen distinto: ¿más cortas? ¿más rápidas? ¿más legibles? ¿por qué?
